# Problem-sequence review (multi-day × multi-session)

Judgment unit is a **stretch** — calendar span × session band — not one `session_num`.
Seeded from date-trusted `text_inside` brackets (largest unaligned mass first).

Look at openings, short unaligned crumbs, day strip (* = no production pin), and
IIIF at stretch ends. Note what looks **local** vs **wall-artefact**.

Actions: **Save note** only (flags: seen / walls_feel_wrong / opening_cluster / other).
No production `PINNED_TO` wire. Persists to `problem_sequence_review_notes`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from IPython.display import HTML, clear_output, display
import ipywidgets as w

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sequence_graph.problem_sequence_review import (
    build_review_deck,
    card_html,
    load_notes,
    record_note,
    save_notes,
)

print("repo root:", ROOT)

In [ ]:
# Full deck = all text_inside seeds (14). Use limit= for a quick smoke.
DECK = build_review_deck()
NOTES = load_notes()
print(
    f"deck={len(DECK)}  prior_notes={len(NOTES)}  "
    f"top unaligned={[c['n_unaligned'] for c in DECK[:5]]}"
)

In [ ]:
flag_box = w.Dropdown(
    options=["seen", "walls_feel_wrong", "opening_cluster", "other"],
    value="seen",
    description="Flag",
)
filter_box = w.Dropdown(
    options=["all", "unnoted", "noted"],
    value="all",
    description="Filter",
)
idx = w.IntSlider(value=0, min=0, max=max(len(DECK) - 1, 0), description="Card")
note_box = w.Textarea(
    value="",
    placeholder="Local cluster? False walls at pins? Openings mid-stretch? …",
    layout=w.Layout(width="100%", height="120px"),
    description="Note",
)
status = w.HTML(value="")
card_out = w.Output()


def _filtered() -> list:
    f = filter_box.value
    if f == "unnoted":
        return [c for c in DECK if c["review_id"] not in NOTES]
    if f == "noted":
        return [c for c in DECK if c["review_id"] in NOTES]
    return DECK


def _sync_slider(*_):
    cards = _filtered()
    idx.max = max(len(cards) - 1, 0)
    if idx.value > idx.max:
        idx.value = idx.max
    _render()


def _current():
    cards = _filtered()
    if not cards:
        return None
    return cards[min(idx.value, len(cards) - 1)]


def _render(*_):
    card = _current()
    with card_out:
        clear_output(wait=True)
        if card is None:
            display(HTML("<p>No cards in this filter.</p>"))
            note_box.value = ""
            return
        prior = NOTES.get(card["review_id"])
        display(HTML(card_html(card, prior)))
        note_box.value = (prior or {}).get("note") or ""
        if prior and prior.get("flag") in dict(flag_box.options):
            flag_box.value = prior["flag"]
        noted = sum(1 for c in DECK if c["review_id"] in NOTES)
        status.value = (
            f"<b>{idx.value + 1}/{len(_filtered())}</b> in filter · "
            f"{noted}/{len(DECK)} noted overall · "
            f"unaligned={card.get('n_unaligned')} openings={card.get('n_opening_hits')}"
        )


def _save(_=None):
    card = _current()
    if card is None:
        return
    record_note(NOTES, card, note_box.value.strip(), flag=flag_box.value)
    save_notes(NOTES)
    cards = _filtered()
    if idx.value < len(cards) - 1:
        idx.value += 1
    else:
        _render()


btn_save = w.Button(description="Save note", button_style="success")
btn_prev = w.Button(description="← Prev")
btn_next = w.Button(description="Next →")
btn_save.on_click(_save)
btn_prev.on_click(lambda _: setattr(idx, "value", max(idx.value - 1, 0)) or _render())
btn_next.on_click(
    lambda _: setattr(idx, "value", min(idx.value + 1, idx.max)) or _render()
)
filter_box.observe(_sync_slider, names="value")
idx.observe(_render, names="value")

ui = w.VBox([
    w.HBox([filter_box, flag_box, idx, status]),
    card_out,
    note_box,
    w.HBox([btn_save, btn_prev, btn_next]),
])
display(ui)
_sync_slider()

## Saved notes

Re-run after a pass.

In [ ]:
import pandas as pd

rows = list(NOTES.values())
if not rows:
    print("No notes yet.")
else:
    df = pd.DataFrame(rows)
    cols = [
        c
        for c in (
            "flag",
            "inventory",
            "calendar_lo",
            "calendar_hi",
            "n_unaligned",
            "n_opening_hits",
            "note",
            "decided_at",
        )
        if c in df.columns
    ]
    display(df[cols].sort_values(["inventory", "calendar_lo"], na_position="last"))
    print(df["flag"].value_counts().to_dict())